# vitok 02 — Pretrain + evaluate (GPU T4 ×2)

**Settings:** Accelerator = GPU T4 ×2, Internet = On. Attach datasets `vitok-data` (from notebook 01) and `vitok-code` (or set `VITOK_REPO`).

Each GPU runs its own queue of conditions, one after another. Run with **Save Version → Save & Run All**; results land in `/kaggle/working/results/` and checkpoints in `/kaggle/working/runs/`.

| Plan step | DEPTH | SEED | QUEUES | SMOKE_ITERS |
|---|---|---|---|---|
| 7 smoke test (Gate 0) | 6, then 8 and 10 | 0 | one condition | 200 (d6), 50 (d8/d10) |
| 8 d6 | 6 | 0 | all 4 split over 2 GPUs | None |
| 9 d8 | 8 | 0, then 1 for bpe-nfc + super-nfc | all 4 | None |
| 10 d10 | 10 | 0 | the 2 chosen at Gate 3 | None |

In [ ]:
DEPTH = 6
SEED = 0
QUEUES = {0: ["bpe-nfc", "bpe-nfd"], 1: ["super-nfc", "super-nfd"]}  # GPU -> conditions
VOCAB = "16k"            # chosen at Gate 1
SMOKE_ITERS = None       # e.g. 200 for the Gate 0 smoke test
DEVICE_BATCH = None      # override per-GPU micro batch on OOM (must divide 64)
NO_COMPILE = False       # set True if torch.compile fails on T4
VITOK_REPO = ""          # leave empty when the vitok-code dataset is attached
NANOCHAT_COMMIT = "92d63d4e8bb4df75c3b71618f31ddde2378b2bcd"

In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

WORK = Path("/kaggle/working")
DATA = next(p.parent for p in Path("/kaggle/input").rglob("test.jsonl") if (p.parent / "shards").exists())
CODE = Path("/tmp/vitok")
bundles = [p.parent for p in Path("/kaggle/input").rglob("pyproject.toml") if (p.parent / "src" / "vitok").exists()]
if bundles:
    shutil.copytree(bundles[0], CODE, dirs_exist_ok=True)
else:
    assert VITOK_REPO, "attach the vitok-code dataset or set VITOK_REPO"
    sh(f"git clone --depth 1 {VITOK_REPO} {CODE}")
NANOCHAT = Path("/tmp/nanochat")
if not NANOCHAT.exists():
    sh(f"git clone https://github.com/karpathy/nanochat {NANOCHAT}")
    sh(f"cd {NANOCHAT} && git checkout {NANOCHAT_COMMIT} && git apply {CODE}/patches/nanochat.patch")
sh(f"pip install -q -e {CODE} tiktoken rustbpe wandb jinja2 pyyaml")
print("data:", DATA)

In [ ]:
# Environment checks: T4 = capability (7, 5), two GPUs, stock tokenizers load every tokenizer.json
import torch
from vitok.hf_tokenizer import HFTokenizer
print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("capability", torch.cuda.get_device_capability(0))
test_docs = [json.loads(l)["text"] for l in open(DATA / "test.jsonl")][:200]
for cond in [c for q in QUEUES.values() for c in q]:
    tok = HFTokenizer.from_directory(DATA / f"tokenizers-{VOCAB}" / cond)
    assert all(tok.decode(ids) == d for ids, d in zip(tok.encode(test_docs), test_docs)), cond
print("tokenizers OK")

In [ ]:
# Launch one queue per GPU and wait. Logs: /kaggle/working/runs/<tag>/train.log, eval.log
procs = {}
for gpu, conds in QUEUES.items():
    if not conds:
        continue
    cmd = [sys.executable, "-m", "vitok.kaggle_run", "--gpu", str(gpu), "--conditions", *conds,
           "--depth", str(DEPTH), "--seed", str(SEED), "--data", str(DATA),
           "--tokenizers", f"tokenizers-{VOCAB}", "--compression", f"compression-{VOCAB}.json",
           "--nanochat", str(NANOCHAT), "--work", str(WORK)]
    if SMOKE_ITERS: cmd += ["--num-iterations", str(SMOKE_ITERS), "--eval-every", "100"]
    if DEVICE_BATCH: cmd += ["--device-batch-size", str(DEVICE_BATCH)]
    if NO_COMPILE: cmd += ["--no-compile"]
    procs[gpu] = subprocess.Popen(cmd, stdout=open(WORK / f"queue_gpu{gpu}.log", "w"), stderr=subprocess.STDOUT)

def tail(path, n=2):
    try:
        return open(path).read().splitlines()[-n:]
    except FileNotFoundError:
        return []

while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    for log in sorted((WORK / "runs").glob(f"*_d{DEPTH}_s{SEED}/train.log")):
        print(log.parent.name, "|", *tail(log, 1))
for gpu, p in procs.items():
    print(f"GPU{gpu} exit={p.returncode}", *tail(WORK / f"queue_gpu{gpu}.log", 5), sep="\n")
assert all(p.returncode == 0 for p in procs.values()), "a queue failed: read queue_gpu*.log and runs/*/train.log"

In [ ]:
# Throughput (Gate 0) and results so far in this session
for log in sorted((WORK / "runs").glob("*/train.log")):
    steps = [l for l in open(log) if l.startswith("step ")]
    print(log.parent.name, "|", steps[-1].strip() if steps else "no steps")
    for l in open(log):
        if l.startswith(("Parameter counts", "transformer_matrices", "lm_head", "total ", "Peak memory", "COMPUTE_DTYPE", "Minimum validation")):
            print("   ", l.strip())
sh(f"python -m vitok.analysis --results {WORK}/results --compression {DATA}/compression-{VOCAB}.json --out {WORK}/results/summary_d{DEPTH}_s{SEED}.md")